In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [2]:
data = pd.read_csv('/Users/krishvenigalla/Desktop/data/cleaned_data.csv')
data.head()

/var/folders/fm/08v38d894qqg0x2sfcmb_2mh0000gn/T/ipykernel_97844/3892212671.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Users/krishvenigalla/Desktop/data/cleaned_data.csv')


,code,product_name,brands,categories_en,labels_en,ingredients_text,allergens_en,additives_en,nutrition_grade_fr,energy_100g,...,cocoa_100g,carbon-footprint_100g,nutrition-score-fr_100g,nutrition-score-uk_100g,category_level_1,category_level_2,category_level_3,category_level_4,category_level_5,category_level_6
0,4530,Banana Chips Sweetened (Whole),not mentioned,NaN,No labels,"Bananas, vegetable oil (coconut oil, corn oil ...",0,No additives,d,2243.0,...,0.0,no information,14.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
1,4559,Peanuts,torn & glasser,NaN,No labels,"Peanuts, wheat flour, sugar, rice flour, tapio...","en:wheat, en:peanuts, en:soy",No additives,b,1941.0,...,0.0,no information,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,16087,Organic Salted Nut Mix,grizzlies,NaN,No labels,"Organic hazelnuts, organic cashews, organic wa...",0,No additives,d,2540.0,...,0.0,no information,12.0,12.0,NaN,NaN,NaN,NaN,NaN,NaN
3,16094,Organic Polenta,bob's red mill,NaN,No labels,Organic polenta,0,No additives,not given,1552.0,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN
4,16100,Breadshop Honey Gone Nuts Granola,unfi,NaN,No labels,"Rolled oats, grape concentrate, expeller press...",en:sesame,No additives,not given,1933.0,...,0.0,no information,not given,not given,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
from modules.ingredients import clean_ingredients

data['ingredients'] = data['ingredients_text'].copy()
data['ingredients_text'] = data['ingredients_text'].apply(clean_ingredients)

### Pre-processing 

In [4]:
# Clean and standardize text columns
def clean_text(text):
    if isinstance(text, str):
        return text.lower().strip()
    return text

data['ingredients'] = data['ingredients'].apply(clean_text)
data['category_level_1'] = data['category_level_1'].apply(clean_text)
data['category_level_2'] = data['category_level_2'].apply(clean_text)

# Replace placeholders with NaN or empty lists
data['ingredients'] = data['ingredients'].replace('ingredients are missing', np.nan)

# Split columns into lists
data['ingredients'] = data['ingredients'].str.split(', ')


### Implement product matching

In [ ]:
# from fuzzywuzzy import process

# def find_top_matches(user_input, choices, limit=5):
#     matches = process.extract(user_input, choices, limit=limit)
#     return matches

In [ ]:
# # Example: User inputs a product name
# user_input = "dark chocolate bar"
# top_matches = find_top_matches(user_input, data['product_name'].tolist())

# print(f"Top matches for '{user_input}':")
# for match, score in top_matches:
#     print(f"- {match} (Score: {score})")

### Tokenization, Stop word removal & Lemmatization

In [6]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [7]:
from modules.ingredients import clean_ingredients
sample_text = "Bananas, vegetable oil (coconut oil, corn oil and/or palm oil) sugar, natural banana flavor." 
sample_text = clean_ingredients(sample_text)
sample_text = re.sub(r'\b(and|or|and/or)\b|[^\w\s]', '', sample_text, flags=re.IGNORECASE).strip()
tokens = word_tokenize(sample_text)
print("Tokens:", tokens)

Tokens: ['bananas', 'vegetable', 'oil', 'sugar', 'natural', 'banana', 'flavor']


In [8]:
stop_words = set(stopwords.words('english'))
tokens = [word for word in tokens if word not in stop_words]
print("Tokens without stop words:", tokens)

Tokens without stop words: ['bananas', 'vegetable', 'oil', 'sugar', 'natural', 'banana', 'flavor']


In [9]:
lemmatizer = WordNetLemmatizer()
tokens = [lemmatizer.lemmatize(word) for word in tokens]
print(tokens)

['banana', 'vegetable', 'oil', 'sugar', 'natural', 'banana', 'flavor']


In [10]:
final_text = ' '.join(tokens)

In [11]:
final_text

'banana vegetable oil sugar natural banana flavor'

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode the text
encoded_text = model.encode(final_text)
print("Encoded text:", encoded_text)

### Recommendations Example Usage using product_name

In [23]:
from modules.ingredient_recommendations import recommend_products
from modules.ingredient_recommendations import recommend_by_ingredients

# Example usage
user_input = "gourmet chicken patties"  # User's input product name
allergens_to_avoid = ['milk']  # Allergens to avoid

recommendations = recommend_products(user_input, data, top_n=5, allergens_to_avoid=allergens_to_avoid)

# Print recommendations
print("Final recommendations:")
print("Returned recommendation columns:", recommendations.columns)
print("Recommendation data:\n", recommendations)
#print(recommendations[['product_name', 'additives_en', 'allergens_en']].to_string(index=False))


Final recommendations:
Returned recommendation columns: Index(['product_name', 'additives_en', 'allergens_en'], dtype='object')
Recommendation data:
                                          product_name  \
477              Premium Chunk White Chicken In Broth   
888   Red Hots Imitation Sausage-Artificially Colored   
1189                              Chicken Noodle Soup   

                      additives_en allergens_en  
477                   No additives            0  
888                 E316,E339,E621   wheat, soy  
1189  E101,E101i,E160a,E160ai,E375  wheat, eggs  


### Recommendations Example Usage using bar_code

In [19]:
from modules.recommendations_code import recommend_products
from modules.recommendations_code import recommend_by_ingredients

# Example usage
bar_code = 16100 
allergens_to_avoid = ['peanuts']  

recommendations = recommend_products(bar_code, data, top_n=5, allergens_to_avoid=allergens_to_avoid)

# Check if recommendations is None
if recommendations is None or recommendations.empty:
	print("No recommendations found. Please check the input data or function implementation.")
else:
	# Print recommendations
	print("Final recommendations:")
	print(recommendations[['product_name', 'additives_en', 'allergens_en']].to_string(index=False))

Final recommendations:
                             product_name additives_en allergens_en
        Breadshop Honey Gone Nuts Granola No additives       sesame
                     Granola Nuts N Honey No additives       sesame
              Fruit & Nut Supreme Granola    E300,E330            0
Sweet P's Bake Shop, Seed & Fruit Granola No additives            0
                      Fruit & Nut Granola No additives            0


### Recommendations Example Usage using bar_code with name weightage

In [22]:
from modules.recommendations import recommend_products 
recommend_products(bar_code=16100, df=data, top_n=5, allergens_to_avoid=['sesame'], name_weight=0.2, match_boost=0.7)

,product_name,additives_en,allergens_en
9482,Fruit & Nut Granola,No additives,0
147492,Honey Almond Granola,No additives,"tree nuts, wheat"
147493,Honey Almond Granola,No additives,"tree nuts, wheat"
48880,"100% Natural Granola, Oats & Honey With Raisins",E422,"wheat, milk"
94721,Granola,No additives,0


### Radial chart

In [ ]:
data.info()

In [ ]:
data.shape

In [ ]:
#data.isna().sum().sort_values(ascending=False).head(20)
data.apply(lambda col: (col == 0).sum()).sort_values(ascending=False).head(25)

In [ ]:
# Select numeric columns (float/int)
numeric_cols = data.select_dtypes(include=['float64', 'int64']).columns

# Drop near-constant numeric columns (95% zeros)
numeric_data = data[numeric_cols]
numeric_data = numeric_data.loc[:, (numeric_data != 0).mean() > 0.05]  # Keep columns with >5% non-zeros

In [ ]:
numeric_data.fillna(0, inplace=True)

In [ ]:
variances = numeric_data.var()
print(variances.sort_values())

In [ ]:
# Remove columns with low variance
numeric_data = numeric_data.drop(columns=['vitamin-a_100g', 'iron_100g', 'vitamin-c_100g', 'cholesterol_100g', 'calcium_100g'])

In [ ]:
# Standardize (mean=0, variance=1)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(numeric_data)

Correlation Heatmap

In [ ]:
import plotly.express as px
import pandas as pd

# Compute correlations
corr_matrix = numeric_data.corr()

# Convert to a DataFrame for Plotly
corr_df = corr_matrix.reset_index().melt(id_vars='index')

# Create interactive heatmap
fig = px.imshow(
    corr_matrix,
    labels=dict(x="Features", y="Features", color="Correlation"),
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1
)

# Add title
fig.update_layout(title="Interactive Correlation Matrix", width=800, height=800)

# Show figure
fig.show()

In [ ]:
data[data['category_level_1']=='salty snacks'][['code', 'fiber_100g']]

In [ ]:
data['fiber_100g'].describe()

In [ ]:
data[data['category_level_1']=='salty snacks'][['fiber_100g', 'salt_100g']].describe()

In [ ]:
data

### Radial chart

In [ ]:
import pandas as pd
from modules.radar_chart import preprocess_data, create_radar_chart, create_radar_chart_with_dropdown

# Define the columns to analyze
category_col = 'category_level_1'
nutrient_cols = ['proteins_100g', 'carbohydrates_100g', 'fiber_100g', 'fat_100g', 'salt_100g']

# Preprocess the data
top_category_nutrition = preprocess_data(data, category_col, nutrient_cols, top_n=20)

# Extract categories and values
categories = top_category_nutrition[category_col]
values = top_category_nutrition[nutrient_cols]

# Create a radar chart
# create_radar_chart(categories, values, title='Top 10 Primary Categories by Nutritional Facts')
create_radar_chart_with_dropdown(categories, values, title='Top 10 Primary Categories by Nutritional Facts')


# Do not run the code from here

## Ingredient based recommendations using tokenization, embeddings and clustering

### Pre-processing and Vectorization

In [12]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk 
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')  
nltk.download('omw-1.4')

ingredients = data.ingredients_text 

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/krishvenigalla/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/krishvenigalla/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/krishvenigalla/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/krishvenigalla/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/krishvenigalla/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


### Step 1 : Preprocessing with Lemmatization

In [13]:
print(type(ingredients[0]))
print(ingredients[0])

<class 'str'>
bananas, vegetable oil sugar, natural banana flavor


In [14]:
def preprocess(text):
    
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

cleaned_ingredients = [preprocess(item) for item in ingredients]

print("Cleaned Ingredients:")
cleaned_ingredients[:5]

Cleaned Ingredients:


['banana vegetable oil sugar natural banana flavor',
 'peanut wheat flour sugar rice flour tapioca starch salt leavening soy sauce potato starch',
 'organic hazelnut organic cashew organic walnut almond organic sunflower oil sea salt',
 'organic polenta',
 'rolled oat grape concentrate expeller pressed canola oil sunflower seed almond walnut oat bran sesame seed cashew natural vitamin e']

### Step 2 : Generate Embeddings with Sentence Transformers

In [15]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the cleaned and lemmatized data
ingredient_embeddings = model.encode(cleaned_ingredients, show_progress_bar=True)

# Print the shape of the embeddings
print(f"Embeddings shape: {ingredient_embeddings.shape}")

Batches:   0%|          | 0/9040 [00:00<?, ?it/s]

Embeddings shape: (289257, 384)


In [16]:
product_name_embeddings = model.encode(data['product_name'].tolist(), show_progress_bar=True)
print(f"Product name embeddings shape: {product_name_embeddings.shape}") 

Batches:   0%|          | 0/9040 [00:00<?, ?it/s]

Product name embeddings shape: (289257, 384)


### Step 3 : Clustering with K-Means (this is a different method)

In [ ]:
# from sklearn.cluster import KMeans

# k = 10  
# kmeans = KMeans(n_clusters=k, random_state=42)
# cluster_labels = kmeans.fit_predict(ingredient_embeddings)

# # Map cluster labels back to ingredients
# clustered_ingredients = pd.DataFrame({
#     'Ingredient': ingredients,
#     'Cleaned_Ingredient': cleaned_ingredients,
#     'Cluster': cluster_labels
# })


# print("\nClustered Data:")
# print(clustered_ingredients)

### Step 3 : Similarity search with FAISS

In [ ]:
import faiss
import numpy as np

# Step 1: Normalize embeddings (critical for cosine similarity)
ingredient_embeddings_normalized = ingredient_embeddings / np.linalg.norm(ingredient_embeddings, axis=1, keepdims=True)

# Step 2: Create FAISS index (cosine similarity = Inner Product on normalized vectors)
dimension = ingredient_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # Inner Product ≈ Cosine Similarity
index.add(ingredient_embeddings_normalized.astype('float32'))

# Step 3: Query (normalize the query too!)
query = "dark chocolate bar"
query_embedding = model.encode([query])
query_embedding_normalized = query_embedding / np.linalg.norm(query_embedding, axis=1, keepdims=True)
k = 5
distances, indices = index.search(query_embedding_normalized.astype('float32'), k)

# Step 4: Map results to ingredients
results = [(ingredients[idx], distances[0][i]) for i, idx in enumerate(indices[0])]
print(f"Top {k} matches for '{query}': {results}")

### Save the embeddings

In [17]:
import numpy as np

# Save embeddings to a file
np.save('/Users/krishvenigalla/Desktop/embeddings/ingredient_embeddings.npy', ingredient_embeddings)
np.save('/Users/krishvenigalla/Desktop/embeddings/product_name_embeddings.npy', product_name_embeddings) 